# Train the V1 representation model

Production training workflow for the variable-length representation learning model. The model learns normal context consistency with EMA latent prediction and normal population geometry with file-level contrastive alignment.

Key features:

- Direct parameters: configure training directly in the notebook via `TrainingParams` class arguments (e.g. `params = TrainingParams(batch_size=32, epochs=10, in_memory=True)`).
- In-memory caching & streaming: set `in_memory=True` (recommended for $\le 25\text{K}$ files) to preload samples into Host RAM with full per-epoch shuffling, or `in_memory=False` for $\mathcal{O}(1)$ disk streaming on $100\text{K}+$ datasets.
- Progress tracking: integrated `tqdm` progress bars for RAM preloading, batch training steps, validation, and reference bank fitting.
- Hardware: automatically accelerates training with CUDA when available (`torch.cuda.is_available()`), falling back to CPU.
- Training dynamics: trains with AdamW optimizer, cosine annealing learning rate scheduler, gradient clipping, EMA target updates, and progressive contrastive ramp-up.
- Checkpointing: atomically saves the trained model, optimizer, scheduler, step count, and normal reference bank to `checkpoints/v1_representation.pt`.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from itertools import islice
import json
import os
from pathlib import Path
import sys
import torch
from tqdm.auto import tqdm

# Explicit Kaggle dataset src path
candidates = [
    Path('/kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01/src'),
    Path('/kaggle/input/anomaly-representation-20260903-01/src'),
    *Path('/kaggle/input').glob('**/src'),
    Path.cwd() / 'src',
    Path.cwd().parent / 'src',
]
for candidate in candidates:
    if (candidate / 'representation').is_dir():
        sys.path.insert(0, str(candidate))
        print(f"Loaded Kaggle representation modules from: {candidate}")
        break

from representation import V1Config
from representation.checkpoint import save_checkpoint
from representation.data import FileDataset, collate_variable_files
from representation.inference import NormalReferenceBank, RepresentationInference
from representation.model import V1RepresentationModel
from representation.trainer import RepresentationTrainer
from synth.config import PatchConfig
from synth.patchify import Patchifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Compute device:', device)
if device.type == 'cuda':
    print('CUDA device name:', torch.cuda.get_device_name(0))
    print('Allocated memory:', f"{torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")

In [ ]:
def _kaggle_data_root() -> str:
    if 'V1_DATA_ROOT' in os.environ:
        return os.environ['V1_DATA_ROOT']
    # Exact path specified by user on Kaggle
    kaggle_path = Path('/kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01')
    if (kaggle_path / 'manifest.json').is_file() or (kaggle_path / 'train').is_dir():
        return str(kaggle_path)
    alt_path = Path('/kaggle/input/anomaly-representation-20260903-01')
    if (alt_path / 'manifest.json').is_file():
        return str(alt_path)
    if Path('/kaggle/input').is_dir():
        for p in Path('/kaggle/input').glob('**/manifest.json'):
            return str(p.parent)
    return 'data/generated/production'

def _kaggle_checkpoint_path() -> str:
    if 'V1_CHECKPOINT_PATH' in os.environ:
        return os.environ['V1_CHECKPOINT_PATH']
    if Path('/kaggle/working').is_dir():
        return '/kaggle/working/v1_representation.pt'
    return 'checkpoints/v1_representation.pt'

@dataclass
class TrainingParams:
    """Kaggle GPU training configuration parameters.
    
    Modify these parameters directly here or pass keyword arguments to TrainingParams(...).
    """
    # Dataset and paths on Kaggle
    data_root: str = _kaggle_data_root()
    checkpoint_path: str = _kaggle_checkpoint_path()
    max_samples: int | None = int(os.environ['V1_MAX_SAMPLES']) if 'V1_MAX_SAMPLES' in os.environ else None
    
    # In-memory RAM caching (fastest on Kaggle 30GB host RAM)
    in_memory: bool = True
    
    # Training hyperparameters (optimized for Kaggle T4 GPU: B=128)
    batch_size: int = 128
    epochs: int = 10
    lr: float = 1.5e-3
    weight_decay: float = 1e-4
    eta_min: float = 1e-5
    max_grad_norm: float = 1.0
    
    # Model architecture
    d_model: int = 128
    sequence_layers: int = 4
    attention_heads: int = 4
    dropout: float = 0.1
    contrastive_ramp_steps: int = 196
    contrastive_warmup_steps: int = 0
    
    # Normal reference bank fitting
    max_ref_samples: int = 8192

# Direct instantiation on Kaggle
params = TrainingParams()

configured_root = Path(params.data_root).expanduser()
repo_root = Path.cwd()
DATA_ROOT = configured_root if configured_root.is_absolute() else repo_root / configured_root
MANIFEST_PATH = DATA_ROOT / 'manifest.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {MANIFEST_PATH}. Run uv run python -m synth.cli --output {DATA_ROOT} first.")
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_splits = ('train', 'val', 'test')
splits = manifest.get('splits')
if not isinstance(splits, dict):
    raise RuntimeError(f"V1 dataset manifest at {MANIFEST_PATH} has no split mapping; regenerate with uv run python -m synth.cli.")
missing_splits = [name for name in required_splits if name not in splits]
if missing_splits:
    raise RuntimeError(f"V1 dataset at {DATA_ROOT} is missing required splits: {', '.join(missing_splits)}. Regenerate with uv run python -m synth.cli.")

def load_split(name: str, limit: int = 4):
    entry = splits[name]
    if not isinstance(entry, dict) or entry.get('status') != 'complete':
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is not complete; rerun uv run python -m synth.cli --output {DATA_ROOT} --resume.")
    samples = list(islice(FileDataset(DATA_ROOT, split=name), limit))
    if not samples:
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is empty.")
    return samples

train_samples = load_split('train')
val_samples = load_split('val')
print('dataset root', DATA_ROOT, 'manifest counts', manifest['counts'])
print('train IDs', [sample.file_id for sample in train_samples], 'val IDs', [sample.file_id for sample in val_samples])
print(f"Configured Kaggle parameters: epochs={params.epochs}, batch_size={params.batch_size}, lr={params.lr}, in_memory={params.in_memory}")

In [ ]:
cfg = V1Config(
    n_channels=train_samples[0].C,
    patch_size=32,
    stride=16,
    d_model=params.d_model,
    sequence_layers=params.sequence_layers,
    attention_heads=params.attention_heads,
    dropout=params.dropout,
    contrastive_warmup_steps=params.contrastive_warmup_steps,
    contrastive_ramp_steps=params.contrastive_ramp_steps,
)
patchifier = Patchifier(PatchConfig(patch_size=32, stride=16, pad_end=True))

class StreamingBatchDataset:
    """Yield collated minibatches, with optional Host RAM caching and full shuffling."""
    def __init__(self, data_root, split, patchifier, config, b_size=32, max_count=None, base_seed=0, in_memory=True):
        self.data_root = data_root
        self.split = split
        self.patchifier = patchifier
        self.config = config
        self.batch_size = b_size
        self.max_count = max_count
        self.base_seed = base_seed
        self.in_memory = in_memory
        self.samples = None
        
        if in_memory:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            total = manifest['counts'][split] if self.max_count is None else min(self.max_count, manifest['counts'][split])
            self.samples = list(tqdm(iterator, total=total, desc=f"Loading {split} to RAM"))

    def __iter__(self):
        if self.samples is not None:
            # In-memory mode: shuffle indices each epoch for true global random shuffling
            indices = torch.randperm(len(self.samples)).tolist()
            chunk = []
            batch_idx = 0
            for idx in indices:
                chunk.append(self.samples[idx])
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )
        else:
            # Streaming mode: stream from disk shards in O(1) RAM
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            chunk = []
            batch_idx = 0
            for sample in iterator:
                chunk.append(sample)
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )

train_batches = StreamingBatchDataset(DATA_ROOT, 'train', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=3, in_memory=params.in_memory)
val_batches = StreamingBatchDataset(DATA_ROOT, 'val', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=4, in_memory=params.in_memory)

train_batch = next(iter(train_batches))
val_batch = next(iter(val_batches))
train_total = manifest['counts']['train'] if params.max_samples is None else min(params.max_samples, manifest['counts']['train'])
val_total = manifest['counts']['val'] if params.max_samples is None else min(params.max_samples, manifest['counts']['val'])
print(f"Datasets ready: train={train_total} files, val={val_total} files, batch_size={params.batch_size}, in_memory={params.in_memory}")
print('train signals', tuple(train_batch['signals'].shape), 'validation signals', tuple(val_batch['signals'].shape), 'mask composition', train_batch['mask_composition'])

In [ ]:
model = V1RepresentationModel(cfg, patchifier=patchifier)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=params.lr, weight_decay=params.weight_decay)

num_batches_per_epoch = (train_total + params.batch_size - 1) // params.batch_size
num_val_batches = (val_total + params.batch_size - 1) // params.batch_size
total_steps = max(1, params.epochs * num_batches_per_epoch)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=params.eta_min)

trainer = RepresentationTrainer(
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    seed=cfg.seed,
    max_grad_norm=params.max_grad_norm,
)

print(f"Starting training: {params.epochs} epochs, {num_batches_per_epoch} batches/epoch ({total_steps} total steps), lr={params.lr}, device={device}...")

best_loss = float('inf')
best_state = None
history = []

epoch_pbar = tqdm(range(1, params.epochs + 1), desc="Training epochs")
for epoch in epoch_pbar:
    batch_pbar = tqdm(train_batches, total=num_batches_per_epoch, desc=f"Epoch {epoch:2d}/{params.epochs}", leave=False)
    train_metrics = trainer.train_epoch(batch_pbar)
    metrics = dict(train_metrics)
    
    val_pbar = tqdm(val_batches, total=num_val_batches, desc="Validating", leave=False)
    val_metrics = trainer.validate(val_pbar)
    metrics.update({f"val_{k}": v for k, v in val_metrics.items()})
    
    if val_metrics["joint_loss"] < best_loss:
        best_loss = val_metrics["joint_loss"]
        best_state = deepcopy(model.state_dict())
        
    trainer.history[-1] = metrics
    history.append(metrics)
    
    epoch_pbar.set_postfix({
        "joint": f"{metrics['joint_loss']:.4f}",
        "val_joint": f"{metrics['val_joint_loss']:.4f}",
        "pred": f"{metrics['prediction_loss']:.4f}",
        "contrast": f"{metrics['contrastive_loss']:.4f}",
        "lambda": f"{metrics['lambda']:.3f}",
    })

if best_state is not None:
    model.load_state_dict(best_state)

print('history', history)

# Fit normal reference bank
model.eval()
max_ref = min(params.max_ref_samples, train_total)
ref_loader = StreamingBatchDataset(DATA_ROOT, 'train', patchifier, cfg, b_size=params.batch_size, max_count=max_ref, base_seed=99, in_memory=params.in_memory)
ref_total_batches = (max_ref + params.batch_size - 1) // params.batch_size
ref_embeddings_list = []
ref_pbar = tqdm(ref_loader, total=ref_total_batches, desc=f"Fitting reference bank ({max_ref} normal files)", leave=False)
with torch.no_grad():
    for batch in ref_pbar:
        dev_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        ref_out = model(dev_batch)
        ref_embeddings_list.append(ref_out['file_embedding'].cpu())
ref_embeddings = torch.cat(ref_embeddings_list, dim=0)
bank = NormalReferenceBank(k=min(cfg.knn_k, ref_embeddings.shape[0])).fit(ref_embeddings)

configured_ckpt = Path(params.checkpoint_path).expanduser()
checkpoint_path = configured_ckpt if configured_ckpt.is_absolute() else repo_root / configured_ckpt
save_checkpoint(
    checkpoint_path,
    model,
    optimizer=trainer.optimizer,
    scheduler=trainer.scheduler,
    step=trainer.step,
    reference_bank=bank,
)
print('checkpoint', checkpoint_path, 'step', trainer.step, f"reference bank: {bank.embeddings.shape[0]} normal files")

In [ ]:
model.eval()
ref_batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in train_batch.items()}
with torch.no_grad():
    reference_output = model(ref_batch_device)
if bank.embeddings is None:
    bank.fit(reference_output['file_embedding'])
inference = RepresentationInference(model, bank, patchifier, masking_config=cfg)
scores = inference.score_batch(val_batch)
print('normal train reference rows', bank.embeddings.shape[0], 'lambda', history[-1]['lambda'])
print('validation S_pred', scores['S_pred'].tolist(), 'S_pop', scores['S_pop'].tolist(), 'timestep localization', scores['timestep_scores'][0].tolist())